# Format Marker Genes

Notebook for formatting the marker genes provided by "Pegasus for Single Cell Analysis" (https://github.com/lilab-bcb/pegasus/blob/master/pegasus/annotate_cluster/human_immune_cell_markers.json)

In [3]:
import json

def extract_subtype_markers(node, is_subtype=False):
    """
    Traversiert den JSON-Baum rekursiv und extrahiert Marker für Subtypes.
    """
    simple_markers = {}   # "Subtype": ["Gene1", "Gene2"]
    weighted_markers = {} # "Subtype": {"Gene1": 1.0, "Gene2": 1.0}

    cell_types = node.get("cell_types", [])

    for ct in cell_types:
        name = ct.get("name")
        has_subtypes = "subtypes" in ct and ct["subtypes"] is not None

        # Wenn wir uns bereits in einer Subtype-Ebene befinden oder ein Blatt-Knoten ohne weitere Subtypes sind
        if is_subtype or not has_subtypes:
            genes_list = []
            gene_weights = {}

            for marker_group in ct.get("markers", []):
                weight = marker_group.get("weight", 1.0)
                for gene in marker_group.get("genes", []):
                    genes_list.append(gene)
                    # Falls ein Gen in mehreren Marker-Gruppen vorkommt, Gewichte aufsummieren/aggregieren
                    gene_weights[gene] = gene_weights.get(gene, 0.0) + weight

            if name:
                simple_markers[name] = list(dict.fromkeys(genes_list)) # Duplikate entfernen
                weighted_markers[name] = gene_weights

        # Rekursiver Aufruf für tiefer liegende Subtypen
        if has_subtypes:
            sub_simple, sub_weighted = extract_subtype_markers(ct["subtypes"], is_subtype=True)
            simple_markers.update(sub_simple)
            weighted_markers.update(sub_weighted)

    return simple_markers, weighted_markers


# 1. Load Marker Gene Data
with open('human_immune_cell_markers.json', 'r') as file:
    data = json.load(file)

# 2. Extract Informations
simple_dict, weighted_dict = extract_subtype_markers(data)

# Print found subtypes
subtype_names = list(simple_dict.keys())
print(f"Found {len(subtype_names)} Subtypes", )
print("Celltypes:", subtype_names)

# 3. Write output in json file for further use

# Option A: only gene names
with open("subtype_markers_simple.json", "w", encoding="utf-8") as f:
    json.dump(simple_dict, f, indent=4, ensure_ascii=False)

# Option B: gene names + weights
with open("subtype_markers_weighted.json", "w", encoding="utf-8") as f:
    json.dump(weighted_dict, f, indent=4, ensure_ascii=False)

Found 37 Subtypes
Celltypes: ['CD4 Naive T cell', 'CD4 TCM', 'CD4 TEM', 'T regulatory cell', 'CD4 CTL', 'T follicular helper cell', 'CD8 Naive T cell', 'CD8 TCM', 'CD8 TEM', 'MAIT', 'Gamma-delta T cell', 'CD56-dim NK cell', 'CD56-bright NK cell', 'Pro B cell', 'Pre B cell', 'Naive B cell', 'Memory B cell', 'Dark zone B cell', 'Light zone B cell', 'Plasma cell', 'CD14+ Monocyte', 'CD16+ Monocyte', 'CD1C+ dendritic cell', 'CLEC9A+ dendritic cell', 'Migratory dendritic cell', 'Plasmacytoid dendritic cell', 'Follicular dendritic cell', 'Hematopoietic stem cell', 'Erythroid cell', 'Platelet', 'Pro-Neutrophil', 'Pre-Neutrophil', 'Neutrophil', 'Basophil', 'M1 macrophage', 'M2 macrophage', 'Mast cell']


In [7]:
blood_immune_celltypes = [
    # T- & NK-Zellen
    'CD4 Naive T cell', 
    'CD4 TCM', 
    'CD4 TEM', 
    'T regulatory cell', 
    'CD4 CTL', 
    'CD8 Naive T cell', 
    'CD8 TCM', 
    'CD8 TEM', 
    'MAIT', 
    'Gamma-delta T cell', 
    'CD56-dim NK cell', 
    'CD56-bright NK cell',
    
    # B-Zellen & Plasmazellen
    'Naive B cell', 
    'Memory B cell', 
    'Plasma cell',
    
    # Myeloide Zellen & Dendritische Zellen
    'CD14+ Monocyte', 
    'CD16+ Monocyte', 
    'CD1C+ dendritic cell', 
    'CLEC9A+ dendritic cell', 
    'Plasmacytoid dendritic cell',
    
    # Granulozyten
    'Neutrophil', 
    'Basophil'
]
print(f'Number of blood immune cells: {len(blood_immune_celltypes)}')

Number of blood immune cells: 22


## Filter the markers to the classes of the datasets

In [10]:
def process_marker_dict(input_file_path: dict) -> dict:
    # 1. Mappings für Subtypen auf Zielklassen
    merge_dict = {
        'CD56-dim NK cell': 'NK cell',
        'CD56-bright NK cell': 'NK cell',
        'Natural killer cell': 'NK cell',
        
        'CD4 TCM': 'CD4 Memory T cell',
        #'CD4 TEM': 'CD4 Memory T cell',  # Falls CD4 TEM ebenfalls zu Memory gehören soll
        'CD4 CTL': 'CD4 Memory T cell',
        'T follicular helper cell': 'CD4 Memory T cell',
        
        'CD8 TEM': 'CD8 Memory T cell',
        'CD8 TCM': 'CD8 Memory T cell',
        
        'Dark zone B cell': 'Memory B cell'
    }

    # 2. Exakte Liste der 15 Ziel-Zelltypen
    target_cell_types = {
        'CD8 Memory T cell',
        'CD4 Naive T cell',
        'CD14+ Monocyte',
        'NK cell',
        'CD4 Memory T cell',
        'Naive B cell',
        'CD8 Naive T cell',
        'Memory B cell',
        'MAIT',
        'CD16+ Monocyte',
        'Gamma-delta T cell',
        'T regulatory cell',
        'CD1C+ dendritic cell',
        'Plasma cell',
        'Plasmacytoid dendritic cell'
    }

    cleaned_dict = {}

    with open(input_file_path, 'r', encoding='utf-8') as f:
        input_dict = json.load(f)

    for raw_name, genes in input_dict.items():
        # Ignoriere Meta-Keys oder ungültige Werte
        if raw_name == 'cell_types' or not isinstance(genes, (list, dict)):
            continue

        # Namens-Mapping anwenden
        mapped_name = merge_dict.get(raw_name, raw_name)

        # Prüfen, ob der Zelltyp in der Zielliste liegt
        if mapped_name in target_cell_types:
            
            # --- Fall A: Gewichtiges Format (dict) ---
            if isinstance(genes, dict):
                if mapped_name not in cleaned_dict:
                    cleaned_dict[mapped_name] = {}
                
                # Falls fälschlicherweise vorher eine Liste angelegt wurde (Sicherheitscheck)
                if isinstance(cleaned_dict[mapped_name], list):
                    cleaned_dict[mapped_name] = {g: 1.0 for g in cleaned_dict[mapped_name]}

                for gene, weight in genes.items():
                    # Falls Gen schon vorhanden ist, behalte das höhere Gewicht bei Merges
                    if gene in cleaned_dict[mapped_name]:
                        cleaned_dict[mapped_name][gene] = max(cleaned_dict[mapped_name][gene], weight)
                    else:
                        cleaned_dict[mapped_name][gene] = weight

            # --- Fall B: Einfaches Format (list) ---
            elif isinstance(genes, list):
                if mapped_name not in cleaned_dict:
                    cleaned_dict[mapped_name] = []
                
                # Gene ohne Duplikate hinzufügen
                for gene in genes:
                    if gene not in cleaned_dict[mapped_name]:
                        cleaned_dict[mapped_name].append(gene)

    return cleaned_dict

# Beispiel für den Aufruf:
simple_result_json = process_marker_dict('subtype_markers_simple.json')
weighted_result_json = process_marker_dict('subtype_markers_weighted.json')

# Option A: only gene names
with open("subtype_markers_simple_filtered.json", "w", encoding="utf-8") as f:
    json.dump(simple_result_json, f, indent=4, ensure_ascii=False)

# Option B: gene names + weights
with open("subtype_markers_weighted_filtered.json", "w", encoding="utf-8") as f:
    json.dump(weighted_result_json, f, indent=4, ensure_ascii=False)